# Milestone 5 — Ensembling Multiple Fine-Tuned Models to Cross 0.75

**Author:** Shruti (22f3002548)  
**Project:** Smart MCQ Solver Challenge  
**Target:** MAP@3 ≥ 0.75

---

### Why My Previous Ensemble Failed

In my first attempt, I blended TF-IDF (MAP@3 ~0.30), SBERT (~0.40), Cross-Encoder (~0.50), and DeBERTa (~0.70). The weak models were injecting noise into the strong model's predictions. Even with optimized weights heavily favoring DeBERTa, the weaker models still pulled the score down on questions where DeBERTa was already correct.

### My New Strategy: Ensemble Multiple Strong Models

Instead of combining one strong model with three weak ones, I'll train **multiple fine-tuned models** with different configurations and ensemble them. Each model individually scores close to or above 0.70, so blending them corrects individual mistakes without adding noise.

**My plan:**
1. Train DeBERTa-v3-base with **3 different random seeds** (same config, different initialization → different error patterns)
2. Train DeBERTa-v3-base with a **different max_length and LoRA rank** (captures different information)
3. Train using **K-fold cross-validation** so every training example gets an out-of-fold prediction (no data leakage for weight optimization)
4. Ensemble ONLY these strong models using optimized weights

**Why this works:** Models trained with different seeds learn slightly different decision boundaries. They agree on "easy" questions but disagree on "hard" ones. When they disagree, the majority vote is usually correct.

In [1]:
!pip install peft==0.13.0 accelerate wandb -q

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy.optimize import minimize
from scipy.special import softmax
import wandb
import gc
import os
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

## 1. Load Dataset & Define Evaluation Utilities

We load the same competition dataset and set up:
- **Label mapping** — convert A/B/C/D/E to numeric indices 0–4
- **MAP@3 functions** — our competition metric for evaluation
- **Detailed breakdown** — shows where the model succeeds and fails by position

In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

option_cols = ['A', 'B', 'C', 'D', 'E']
label_to_idx = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
idx_to_label = {v: k for k, v in label_to_idx.items()}

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

def ap_at_3(true_label, predicted_labels):
    for i, pred in enumerate(predicted_labels[:3]):
        if pred.strip().upper() == true_label.strip().upper():
            return 1.0 / (i + 1)
    return 0.0

def map_at_3(true_labels, predicted_labels):
    return np.mean([ap_at_3(t, p) for t, p in zip(true_labels, predicted_labels)])

def map_at_3_detailed(true_labels, predicted_labels):
    scores = [ap_at_3(t, p) for t, p in zip(true_labels, predicted_labels)]
    n = len(scores)
    return {
        'map3': np.mean(scores),
        'correct_at_1': sum(1 for s in scores if s == 1.0),
        'correct_at_2': sum(1 for s in scores if s == 0.5),
        'correct_at_3': sum(1 for s in scores if abs(s - 1/3) < 0.01),
        'missed': sum(1 for s in scores if s == 0.0),
        'total': n,
        'top1_acc': sum(1 for s in scores if s == 1.0) / n,
        'top3_acc': sum(1 for s in scores if s > 0) / n,
    }

def print_results(name, results):
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  MAP@3:          {results['map3']:.4f}")
    print(f"  Top-1 Accuracy: {results['top1_acc']:.2%}")
    print(f"  Top-3 Accuracy: {results['top3_acc']:.2%}")
    print(f"  Correct at #1:  {results['correct_at_1']}/{results['total']}")
    print(f"  Correct at #2:  {results['correct_at_2']}/{results['total']}")
    print(f"  Correct at #3:  {results['correct_at_3']}/{results['total']}")
    print(f"  Missed:         {results['missed']}/{results['total']}")

def logits_to_preds(logits):
    """I convert a (N, 5) logit matrix to a list of top-3 label predictions."""
    preds = []
    for i in range(len(logits)):
        top3 = np.argsort(logits[i])[::-1][:3]
        preds.append([idx_to_label[idx] for idx in top3])
    return preds

print("Setup complete.")

Train: (2000, 8), Test: (500, 7)
Setup complete.


## 2. W&B Login

Login to Weights & Biases for experiment tracking. API key is stored as a Kaggle Secret named `WANDB_API_KEY`.

In [3]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))
print("W&B login successful!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B login successful!


## 3. Dataset Class & Training Infrastructure

I define my MCQ dataset class and a reusable training function. This lets me train multiple models with different configs without repeating code. Each call to `train_and_predict` produces:
- **Train logits:** shape (2000, 5) — scores for every training question
- **Test logits:** shape (500, 5) — scores for every test question

I'll collect logits from every model and blend them at the end.

In [4]:
class MCQDataset(Dataset):
    """My MCQ dataset — formats each question as 5 (prompt, option) pairs."""
    
    def __init__(self, df, tokenizer, max_length=256, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        first_sentences = [prompt] * 5
        second_sentences = [str(row[col]) for col in option_cols]
        
        tokenized = self.tokenizer(
            first_sentences, second_sentences,
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt'
        )
        
        item = {
            'input_ids': tokenized['input_ids'],
            'attention_mask': tokenized['attention_mask'],
        }
        if 'token_type_ids' in tokenized:
            item['token_type_ids'] = tokenized['token_type_ids']
        
        if not self.is_test:
            item['labels'] = torch.tensor(label_to_idx[row['answer']], dtype=torch.long)
        
        return item


class MCQDataCollator:
    def __call__(self, features):
        batch = {
            'input_ids': torch.stack([f['input_ids'] for f in features]),
            'attention_mask': torch.stack([f['attention_mask'] for f in features]),
        }
        if 'token_type_ids' in features[0]:
            batch['token_type_ids'] = torch.stack([f['token_type_ids'] for f in features])
        if 'labels' in features[0]:
            batch['labels'] = torch.stack([f['labels'] for f in features])
        return batch


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds_top1 = np.argmax(logits, axis=1)
    accuracy = (preds_top1 == labels).mean()
    
    predicted_labels = []
    true_labels = []
    for i in range(len(labels)):
        top3 = np.argsort(logits[i])[::-1][:3]
        predicted_labels.append([idx_to_label[idx] for idx in top3])
        true_labels.append(idx_to_label[labels[i]])
    
    return {'accuracy': accuracy, 'map3': map_at_3(true_labels, predicted_labels)}


print("Infrastructure ready.")

Infrastructure ready.


## 4. My Reusable Training Function

This is the core function I'll call multiple times with different configurations. It handles the full pipeline: load model → apply LoRA → train → predict on full train + test → cleanup GPU memory.

I designed it to be memory-efficient: after extracting logits, I delete the model and clear GPU cache so I have room for the next one.

In [5]:
def train_and_predict(
    train_data, val_data, train_df_full, test_df,
    model_name="microsoft/deberta-v3-base",
    max_length=256,
    lora_r=16,
    lora_alpha=32,
    learning_rate=2e-5,
    epochs=10,
    batch_size=2,
    grad_accum=8,
    seed=42,
    run_name="model",
):
    """
    I train a single LoRA model and return logits for the full training set and test set.
    
    Returns:
        train_logits: (len(train_df_full), 5) — logits for ALL training questions
        test_logits: (len(test_df), 5) — logits for all test questions
        val_map3: float — validation MAP@3 score
    """
    print(f"\n{'='*55}")
    print(f"  Training: {run_name}")
    print(f"  seed={seed}, lr={learning_rate}, max_len={max_length}, r={lora_r}")
    print(f"{'='*55}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Create datasets
    train_dataset = MCQDataset(train_data, tokenizer, max_length)
    val_dataset = MCQDataset(val_data, tokenizer, max_length)
    full_train_dataset = MCQDataset(train_df_full, tokenizer, max_length, is_test=False)
    test_dataset = MCQDataset(test_df, tokenizer, max_length, is_test=True)
    
    # Load model + LoRA
    base_model = AutoModelForMultipleChoice.from_pretrained(model_name)
    
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.1,
        target_modules=["query_proj", "value_proj"],
        bias="none",
    )
    
    model = get_peft_model(base_model, lora_config)
    model = model.to(device)
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {trainable:,}")
    
    # Training args
    training_args = TrainingArguments(
        output_dir=f'./output_{run_name}',
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=grad_accum,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.1,
        fp16=False,  # DeBERTa-v3 incompatible with fp16
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=20,
        load_best_model_at_end=True,
        metric_for_best_model="map3",
        greater_is_better=True,
        save_total_limit=1,
        report_to="none",  # I'll log manually after
        seed=seed,
        dataloader_num_workers=2,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=MCQDataCollator(),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    
    # Train
    trainer.train()
    
    # Get validation score
    eval_results = trainer.evaluate()
    val_map3 = eval_results['eval_map3']
    print(f"\n  Val MAP@3: {val_map3:.4f}")
    
    # Get logits for full training set
    print("  Predicting on full train set...")
    train_output = trainer.predict(full_train_dataset)
    train_logits = train_output.predictions
    
    # Get logits for test set
    print("  Predicting on test set...")
    test_output = trainer.predict(test_dataset)
    test_logits = test_output.predictions
    
    # Cleanup GPU memory — critical for training multiple models
    del model, trainer, base_model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU memory cleared. Done with {run_name}.")
    
    return train_logits, test_logits, val_map3


print("Training function ready.")

Training function ready.


## 5. Train Model 1 — DeBERTa-v3, Seed 42, Standard Config

This is my baseline configuration — the same setup I used in Milestone 4. It serves as the anchor of my ensemble. Every other model I train will have some variation that makes it learn slightly different patterns.

In [6]:
# Standard train/val split
train_data_1, val_data_1 = train_test_split(
    train_df, test_size=0.15, random_state=42, stratify=train_df['answer']
)
train_data_1 = train_data_1.reset_index(drop=True)
val_data_1 = val_data_1.reset_index(drop=True)

logits_train_1, logits_test_1, val_score_1 = train_and_predict(
    train_data=train_data_1,
    val_data=val_data_1,
    train_df_full=train_df,
    test_df=test_df,
    max_length=256,
    lora_r=16,
    lora_alpha=32,
    learning_rate=2e-5,
    seed=42,
    run_name="deberta_seed42_r16_len256",
)

# Check standalone performance
preds_1 = logits_to_preds(logits_train_1)
results_1 = map_at_3_detailed(train_df['answer'].tolist(), preds_1)
print_results("Model 1 (seed=42, r=16, len=256)", results_1)


  Training: deberta_seed42_r16_len256
  seed=42, lr=2e-05, max_len=256, r=16


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Trainable parameters: 590,593


Epoch,Training Loss,Validation Loss,Accuracy,Map3
1,25.685645,3.171875,0.300000,0.484444
2,25.676660,3.166016,0.310000,0.511667
3,25.612402,3.162109,0.320000,0.510556
4,25.419238,3.150391,0.340000,0.505000



  Val MAP@3: 0.5117
  Predicting on full train set...
  Predicting on test set...


  GPU memory cleared. Done with deberta_seed42_r16_len256.

  Model 1 (seed=42, r=16, len=256)
  MAP@3:          0.4823
  Top-1 Accuracy: 30.45%
  Top-3 Accuracy: 71.90%
  Correct at #1:  609/2000
  Correct at #2:  475/2000
  Correct at #3:  354/2000
  Missed:         562/2000


## 6. Train Model 2 — Different Seed (123)

Same exact configuration but with a different random seed. This changes:
- Weight initialization of the LoRA adapters
- Order of training batches (data shuffling)
- Dropout patterns during training

These small differences cause the model to converge to a slightly different solution. On "easy" questions both models agree, but on "hard" questions they'll often disagree — which is exactly what I need for a useful ensemble.

In [7]:
train_data_2, val_data_2 = train_test_split(
    train_df, test_size=0.15, random_state=123, stratify=train_df['answer']
)
train_data_2 = train_data_2.reset_index(drop=True)
val_data_2 = val_data_2.reset_index(drop=True)

logits_train_2, logits_test_2, val_score_2 = train_and_predict(
    train_data=train_data_2,
    val_data=val_data_2,
    train_df_full=train_df,
    test_df=test_df,
    max_length=256,
    lora_r=16,
    lora_alpha=32,
    learning_rate=2e-5,
    seed=123,
    run_name="deberta_seed123_r16_len256",
)

preds_2 = logits_to_preds(logits_train_2)
results_2 = map_at_3_detailed(train_df['answer'].tolist(), preds_2)
print_results("Model 2 (seed=123, r=16, len=256)", results_2)


  Training: deberta_seed123_r16_len256
  seed=123, lr=2e-05, max_len=256, r=16


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight              

  Trainable parameters: 590,593


Epoch,Training Loss,Validation Loss,Accuracy,Map3
1,25.762793,3.207031,0.240000,0.407222
2,25.565625,3.201172,0.270000,0.419444
3,25.669727,3.193359,0.256667,0.421111
4,25.503418,3.187500,0.280000,0.428333
5,25.666016,3.181641,0.273333,0.430000
6,25.455273,3.173828,0.263333,0.435000
7,25.717285,3.169922,0.260000,0.436667
8,25.641504,3.164062,0.270000,0.442778
9,25.397852,3.158203,0.273333,0.443889
10,24.280762,3.156250,0.273333,0.441667



  Val MAP@3: 0.4439
  Predicting on full train set...
  Predicting on test set...


  GPU memory cleared. Done with deberta_seed123_r16_len256.

  Model 2 (seed=123, r=16, len=256)
  MAP@3:          0.4768
  Top-1 Accuracy: 30.25%
  Top-3 Accuracy: 71.45%
  Correct at #1:  605/2000
  Correct at #2:  443/2000
  Correct at #3:  381/2000
  Missed:         571/2000


## 7. Train Model 3 — Different Seed (999)

My third seed variation. With three models trained on the same data but different seeds, I can do a robust majority vote. The probability that all three models make the same mistake on a given question is much lower than any single model making that mistake.

In [8]:
train_data_3, val_data_3 = train_test_split(
    train_df, test_size=0.15, random_state=999, stratify=train_df['answer']
)
train_data_3 = train_data_3.reset_index(drop=True)
val_data_3 = val_data_3.reset_index(drop=True)

logits_train_3, logits_test_3, val_score_3 = train_and_predict(
    train_data=train_data_3,
    val_data=val_data_3,
    train_df_full=train_df,
    test_df=test_df,
    max_length=256,
    lora_r=16,
    lora_alpha=32,
    learning_rate=2e-5,
    seed=999,
    run_name="deberta_seed999_r16_len256",
)

preds_3 = logits_to_preds(logits_train_3)
results_3 = map_at_3_detailed(train_df['answer'].tolist(), preds_3)
print_results("Model 3 (seed=999, r=16, len=256)", results_3)


  Training: deberta_seed999_r16_len256
  seed=999, lr=2e-05, max_len=256, r=16


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight              

  Trainable parameters: 590,593


Epoch,Training Loss,Validation Loss,Accuracy,Map3
1,25.773340,3.201172,0.263333,0.438889
2,25.734180,3.193359,0.270000,0.444444
3,25.656641,3.185547,0.313333,0.471111
4,25.611426,3.173828,0.333333,0.486667
5,25.614160,3.160156,0.370000,0.518333
6,25.541699,3.146484,0.353333,0.517778
7,25.361426,3.134766,0.356667,0.522778
8,25.288379,3.123047,0.360000,0.528333
9,25.305664,3.117188,0.363333,0.530000
10,24.203809,3.115234,0.373333,0.534444



  Val MAP@3: 0.5344
  Predicting on full train set...
  Predicting on test set...


  GPU memory cleared. Done with deberta_seed999_r16_len256.

  Model 3 (seed=999, r=16, len=256)
  MAP@3:          0.5140
  Top-1 Accuracy: 34.35%
  Top-3 Accuracy: 74.60%
  Correct at #1:  687/2000
  Correct at #2:  436/2000
  Correct at #3:  369/2000
  Missed:         508/2000


## 8. Train Model 4 — Higher LoRA Rank (r=32) + Longer Context (384 tokens)

Now I change the architecture, not just the seed. A higher LoRA rank gives the adapter more capacity to learn complex patterns. A longer max_length captures more text from prompts and options that were previously truncated.

This model sees the data differently from Models 1–3, which adds true architectural diversity to my ensemble — not just initialization diversity.

In [9]:
train_data_4, val_data_4 = train_test_split(
    train_df, test_size=0.15, random_state=42, stratify=train_df['answer']
)
train_data_4 = train_data_4.reset_index(drop=True)
val_data_4 = val_data_4.reset_index(drop=True)

logits_train_4, logits_test_4, val_score_4 = train_and_predict(
    train_data=train_data_4,
    val_data=val_data_4,
    train_df_full=train_df,
    test_df=test_df,
    max_length=384,       # longer context
    lora_r=32,            # higher rank
    lora_alpha=64,        # 2x rank
    learning_rate=2e-5,
    seed=42,
    run_name="deberta_seed42_r32_len384",
)

preds_4 = logits_to_preds(logits_train_4)
results_4 = map_at_3_detailed(train_df['answer'].tolist(), preds_4)
print_results("Model 4 (seed=42, r=32, len=384)", results_4)


  Training: deberta_seed42_r32_len384
  seed=42, lr=2e-05, max_len=384, r=32


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight              

  Trainable parameters: 1,180,417


Epoch,Training Loss,Validation Loss,Accuracy,Map3
1,26.048145,3.244141,0.176667,0.340556
2,25.738867,3.226562,0.223333,0.388889
3,25.751953,3.207031,0.240000,0.417222
4,25.520801,3.189453,0.276667,0.441111
5,25.433594,3.175781,0.263333,0.451111
6,25.355371,3.162109,0.293333,0.462222
7,25.412207,3.142578,0.296667,0.475000
8,25.273242,3.126953,0.303333,0.475556
9,25.036816,3.121094,0.326667,0.486111
10,23.871191,3.119141,0.316667,0.482222



  Val MAP@3: 0.4861
  Predicting on full train set...
  Predicting on test set...


  GPU memory cleared. Done with deberta_seed42_r32_len384.

  Model 4 (seed=42, r=32, len=384)
  MAP@3:          0.5178
  Top-1 Accuracy: 34.30%
  Top-3 Accuracy: 74.35%
  Correct at #1:  686/2000
  Correct at #2:  495/2000
  Correct at #3:  306/2000
  Missed:         513/2000


## 9. Train Model 5 — Slightly Higher Learning Rate + All 3 Attention Projections

My final model uses a slightly higher learning rate (3e-5) for more aggressive optimization, and applies LoRA to query, key, AND value projections instead of just query and value. Adding the key projection gives the model more trainable parameters and more ways to adapt its attention patterns.

In [10]:
# For this model I need to adjust target_modules
# First check if key_proj exists in DeBERTa-v3
temp_model = AutoModelForMultipleChoice.from_pretrained("microsoft/deberta-v3-base")
key_modules = [n for n, _ in temp_model.named_modules() if 'key_proj' in n]
print(f"Found {len(key_modules)} key_proj layers: {key_modules[:3]}...")
del temp_model
gc.collect()

train_data_5, val_data_5 = train_test_split(
    train_df, test_size=0.15, random_state=777, stratify=train_df['answer']
)
train_data_5 = train_data_5.reset_index(drop=True)
val_data_5 = val_data_5.reset_index(drop=True)

# I need a custom version of train_and_predict with different target_modules
tokenizer_5 = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
base_5 = AutoModelForMultipleChoice.from_pretrained("microsoft/deberta-v3-base")

lora_config_5 = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query_proj", "key_proj", "value_proj"],  # all 3 projections
    bias="none",
)

model_5 = get_peft_model(base_5, lora_config_5)
model_5 = model_5.to(device)
model_5.print_trainable_parameters()

train_dataset_5 = MCQDataset(train_data_5, tokenizer_5, 256)
val_dataset_5 = MCQDataset(val_data_5, tokenizer_5, 256)

training_args_5 = TrainingArguments(
    output_dir='./output_model5',
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=3e-5,       # slightly higher LR
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    seed=777,
    dataloader_num_workers=2,
)

trainer_5 = Trainer(
    model=model_5,
    args=training_args_5,
    train_dataset=train_dataset_5,
    eval_dataset=val_dataset_5,
    data_collator=MCQDataCollator(),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer_5.train()
eval_5 = trainer_5.evaluate()
val_score_5 = eval_5['eval_map3']
print(f"  Val MAP@3: {val_score_5:.4f}")

# Get logits
full_ds_5 = MCQDataset(train_df, tokenizer_5, 256, is_test=False)
test_ds_5 = MCQDataset(test_df, tokenizer_5, 256, is_test=True)

logits_train_5 = trainer_5.predict(full_ds_5).predictions
logits_test_5 = trainer_5.predict(test_ds_5).predictions

preds_5 = logits_to_preds(logits_train_5)
results_5 = map_at_3_detailed(train_df['answer'].tolist(), preds_5)
print_results("Model 5 (seed=777, lr=3e-5, QKV)", results_5)

del model_5, trainer_5, base_5
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight              

Found 12 key_proj layers: ['deberta.encoder.layer.0.attention.self.key_proj', 'deberta.encoder.layer.1.attention.self.key_proj', 'deberta.encoder.layer.2.attention.self.key_proj']...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight              

trainable params: 885,505 || all params: 185,308,418 || trainable%: 0.4779


Epoch,Training Loss,Validation Loss,Accuracy,Map3
1,25.635547,3.185547,0.270000,0.447222
2,25.672461,3.167969,0.250000,0.452778
3,25.563770,3.138672,0.310000,0.491111
4,25.411719,3.111328,0.336667,0.519444
5,25.384961,3.066406,0.366667,0.540000
6,24.994531,3.021484,0.390000,0.558889
7,24.900684,2.982422,0.413333,0.565556
8,24.616699,2.939453,0.413333,0.565556
9,24.358105,2.917969,0.416667,0.568889
10,23.322168,2.908203,0.416667,0.566111


  Val MAP@3: 0.5689



  Model 5 (seed=777, lr=3e-5, QKV)
  MAP@3:          0.5749
  Top-1 Accuracy: 41.60%
  Top-3 Accuracy: 78.10%
  Correct at #1:  832/2000
  Correct at #2:  447/2000
  Correct at #3:  283/2000
  Missed:         438/2000


## 10. Standalone Performance Summary

Before ensembling, I check two things:
1. **Individual scores** — all models should be strong (above 0.65)
2. **Model diversity** — models should disagree on some questions (otherwise ensembling adds nothing)

In [11]:
all_model_names = [
    'M1: seed42, r16, 256',
    'M2: seed123, r16, 256',
    'M3: seed999, r16, 256',
    'M4: seed42, r32, 384',
    'M5: seed777, lr3e-5, QKV',
]
all_train_logits = [logits_train_1, logits_train_2, logits_train_3, logits_train_4, logits_train_5]
all_test_logits = [logits_test_1, logits_test_2, logits_test_3, logits_test_4, logits_test_5]
all_val_scores = [val_score_1, val_score_2, val_score_3, val_score_4, val_score_5]
all_preds = [preds_1, preds_2, preds_3, preds_4, preds_5]

true_labels = train_df['answer'].tolist()

print(f"{'='*55}")
print(f"  Individual Model Performance")
print(f"{'='*55}")
for name, vs, preds in zip(all_model_names, all_val_scores, all_preds):
    train_map3 = map_at_3(true_labels, preds)
    print(f"  {name}")
    print(f"    Val MAP@3:   {vs:.4f}")
    print(f"    Train MAP@3: {train_map3:.4f}")

# Model agreement analysis — I want LOW agreement (= high diversity)
print(f"\n  Agreement Matrix (% same top-1):")
print(f"  {'':>8}", end='')
for i in range(5):
    print(f"{'M'+str(i+1):>8}", end='')
print()

for i in range(5):
    print(f"  {'M'+str(i+1):>8}", end='')
    for j in range(5):
        agree = sum(1 for k in range(len(train_df)) 
                    if all_preds[i][k][0] == all_preds[j][k][0])
        print(f"{agree/len(train_df):>7.1%}", end='')
    print()

# Ensemble opportunity
any_correct = sum(1 for i in range(len(train_df)) 
                  if any(p[i][0] == true_labels[i] for p in all_preds))
all_correct = sum(1 for i in range(len(train_df)) 
                  if all(p[i][0] == true_labels[i] for p in all_preds))
none_correct = len(train_df) - any_correct

print(f"\n  At least one model correct: {any_correct}/{len(train_df)} ({any_correct/len(train_df):.1%})")
print(f"  All models correct:         {all_correct}/{len(train_df)} ({all_correct/len(train_df):.1%})")
print(f"  No model correct:           {none_correct}/{len(train_df)} ({none_correct/len(train_df):.1%})")
print(f"\n  Ensemble ceiling (if I could always pick the right model): {any_correct/len(train_df):.1%}")

  Individual Model Performance
  M1: seed42, r16, 256
    Val MAP@3:   0.5117
    Train MAP@3: 0.4823
  M2: seed123, r16, 256
    Val MAP@3:   0.4439
    Train MAP@3: 0.4768
  M3: seed999, r16, 256
    Val MAP@3:   0.5344
    Train MAP@3: 0.5140
  M4: seed42, r32, 384
    Val MAP@3:   0.4861
    Train MAP@3: 0.5178
  M5: seed777, lr3e-5, QKV
    Val MAP@3:   0.5689
    Train MAP@3: 0.5749

  Agreement Matrix (% same top-1):
                M1      M2      M3      M4      M5
        M1 100.0%  50.2%  48.4%  37.1%  47.2%
        M2  50.2% 100.0%  61.5%  46.4%  60.9%
        M3  48.4%  61.5% 100.0%  52.0%  62.6%
        M4  37.1%  46.4%  52.0% 100.0%  52.6%
        M5  47.2%  60.9%  62.6%  52.6% 100.0%

  At least one model correct: 1263/2000 (63.1%)
  All models correct:         195/2000 (9.8%)
  No model correct:           737/2000 (36.9%)

  Ensemble ceiling (if I could always pick the right model): 63.1%


## 11. Ensemble Method 1 — Simple Average of Logits

My simplest blend: just average the raw logits from all 5 models. Since all models are strong and produce scores on similar scales, this should already beat any individual model.

I average raw logits (not softmax) because raw logits preserve the model's confidence calibration better. Softmax squashes extreme values, which can hide useful signal.

In [12]:
# Stack all logits: (5, 2000, 5)
train_logits_stack = np.stack(all_train_logits)
test_logits_stack = np.stack(all_test_logits)

# Simple average
avg_train_logits = train_logits_stack.mean(axis=0)  # (2000, 5)
avg_test_logits = test_logits_stack.mean(axis=0)

avg_preds = logits_to_preds(avg_train_logits)
results_avg = map_at_3_detailed(true_labels, avg_preds)
print_results("Ensemble — Simple Average (5 models)", results_avg)


  Ensemble — Simple Average (5 models)
  MAP@3:          0.5492
  Top-1 Accuracy: 37.65%
  Top-3 Accuracy: 77.40%
  Correct at #1:  753/2000
  Correct at #2:  482/2000
  Correct at #3:  313/2000
  Missed:         452/2000


## 12. Ensemble Method 2 — Validation-Weighted Average

I weight each model by its validation MAP@3. Models that performed better on their validation set get more influence. This is a natural and principled weighting scheme since validation performance is an honest estimate of generalization ability.

In [13]:
# Weight by validation performance
val_weights = np.array(all_val_scores)
val_weights = val_weights / val_weights.sum()

print("Validation-based weights:")
for name, w, vs in zip(all_model_names, val_weights, all_val_scores):
    print(f"  {name}: weight={w:.3f} (val MAP@3={vs:.4f})")

weighted_train_logits = np.zeros_like(all_train_logits[0])
weighted_test_logits = np.zeros_like(all_test_logits[0])

for i, w in enumerate(val_weights):
    weighted_train_logits += w * all_train_logits[i]
    weighted_test_logits += w * all_test_logits[i]

weighted_preds = logits_to_preds(weighted_train_logits)
results_weighted = map_at_3_detailed(true_labels, weighted_preds)
print_results("Ensemble — Validation-Weighted Average", results_weighted)

Validation-based weights:
  M1: seed42, r16, 256: weight=0.201 (val MAP@3=0.5117)
  M2: seed123, r16, 256: weight=0.174 (val MAP@3=0.4439)
  M3: seed999, r16, 256: weight=0.210 (val MAP@3=0.5344)
  M4: seed42, r32, 384: weight=0.191 (val MAP@3=0.4861)
  M5: seed777, lr3e-5, QKV: weight=0.224 (val MAP@3=0.5689)

  Ensemble — Validation-Weighted Average
  MAP@3:          0.5497
  Top-1 Accuracy: 37.75%
  Top-3 Accuracy: 77.30%
  Correct at #1:  755/2000
  Correct at #2:  485/2000
  Correct at #3:  306/2000
  Missed:         454/2000


## 13. Ensemble Method 3 — Optimized Weights with Scipy

Now I search for the exact weight combination that maximizes MAP@3. I use scipy's Nelder-Mead optimizer since MAP@3 is non-differentiable. I try multiple starting points to avoid local optima.

Since all 5 models are strong, I constrain every weight to be at least 0.05 — no model gets completely zeroed out. This prevents the optimizer from collapsing to a single model.

In [14]:
def neg_map3_with_weights(weights_raw, logits_stack, true_labels):
    """Scipy minimizes, so I negate MAP@3. I use softmax to ensure valid weights."""
    weights = softmax(weights_raw)
    blended = np.zeros_like(logits_stack[0])
    for i, w in enumerate(weights):
        blended += w * logits_stack[i]
    preds = logits_to_preds(blended)
    return -map_at_3(true_labels, preds)

# Multiple starting points
starts = [
    [1, 1, 1, 1, 1],                    # equal
    [1, 1, 1, 2, 1],                    # favor model 4
    [1, 1, 1, 1, 2],                    # favor model 5
    [2, 1, 1, 2, 2],                    # favor 1, 4, 5
    list(val_weights * 5),               # validation-weighted
    [0.5, 0.5, 0.5, 3, 2],             # heavily favor 4 and 5
]

best_opt_score = 0
best_opt_weights = None

print("Optimizing ensemble weights...")
for i, start in enumerate(starts):
    result = minimize(
        neg_map3_with_weights,
        x0=start,
        args=(train_logits_stack, true_labels),
        method='Nelder-Mead',
        options={'maxiter': 1000, 'xatol': 0.0005}
    )
    
    opt_weights = softmax(result.x)
    opt_score = -result.fun
    
    print(f"  Start {i+1}: MAP@3={opt_score:.4f}  weights=[{', '.join(f'{w:.3f}' for w in opt_weights)}]")
    
    if opt_score > best_opt_score:
        best_opt_score = opt_score
        best_opt_weights = opt_weights

print(f"\nBest optimized weights:")
for name, w in zip(all_model_names, best_opt_weights):
    print(f"  {name}: {w:.3f}")
print(f"Best optimized MAP@3: {best_opt_score:.4f}")

Optimizing ensemble weights...
  Start 1: MAP@3=0.5498  weights=[0.190, 0.204, 0.197, 0.204, 0.204]
  Start 2: MAP@3=0.5518  weights=[0.163, 0.151, 0.155, 0.359, 0.171]
  Start 3: MAP@3=0.5712  weights=[0.107, 0.067, 0.052, 0.052, 0.723]
  Start 4: MAP@3=0.5733  weights=[0.213, 0.041, 0.040, 0.095, 0.610]
  Start 5: MAP@3=0.5514  weights=[0.194, 0.171, 0.216, 0.199, 0.220]
  Start 6: MAP@3=0.5752  weights=[0.000, 0.000, 0.000, 0.000, 0.999]

Best optimized weights:
  M1: seed42, r16, 256: 0.000
  M2: seed123, r16, 256: 0.000
  M3: seed999, r16, 256: 0.000
  M4: seed42, r32, 384: 0.000
  M5: seed777, lr3e-5, QKV: 0.999
Best optimized MAP@3: 0.5752


## 14. Ensemble Method 4 — Rank-Based Fusion

One more approach: instead of blending raw logits, I convert each model's scores to **ranks** (1st, 2nd, 3rd, 4th, 5th) and average the ranks. The option with the best average rank wins.

Rank-based fusion is more robust to outlier scores — if one model gives a wildly high logit to a wrong answer, it still only gets rank 1. This prevents any single model's overconfident mistake from dominating the ensemble.

In [15]:
def logits_to_ranks(logits):
    """I convert logits to ranks: 5=best, 1=worst for each question."""
    ranks = np.zeros_like(logits)
    for i in range(len(logits)):
        order = np.argsort(logits[i])
        for rank, idx in enumerate(order):
            ranks[i, idx] = rank + 1  # 1=worst, 5=best
    return ranks

# Convert all models to ranks
all_train_ranks = np.stack([logits_to_ranks(l) for l in all_train_logits])
all_test_ranks = np.stack([logits_to_ranks(l) for l in all_test_logits])

# Average ranks
avg_train_ranks = all_train_ranks.mean(axis=0)
avg_test_ranks = all_test_ranks.mean(axis=0)

rank_preds = logits_to_preds(avg_train_ranks)
results_rank = map_at_3_detailed(true_labels, rank_preds)
print_results("Ensemble — Rank-Based Fusion", results_rank)


  Ensemble — Rank-Based Fusion
  MAP@3:          0.5262
  Top-1 Accuracy: 35.30%
  Top-3 Accuracy: 75.35%
  Correct at #1:  706/2000
  Correct at #2:  477/2000
  Correct at #3:  324/2000
  Missed:         493/2000


## 15. Pick the Best Ensemble & Generate Submission

I compare all ensemble methods and use the best one for my final Kaggle submission.

In [16]:
# Compare all approaches
ensemble_results = {
    'Simple Average': (results_avg, avg_test_logits),
    'Val-Weighted': (results_weighted, weighted_test_logits),
    'Optimized Weights': (
        map_at_3_detailed(true_labels, logits_to_preds(
            sum(w * l for w, l in zip(best_opt_weights, all_train_logits))
        )),
        sum(w * l for w, l in zip(best_opt_weights, all_test_logits))
    ),
    'Rank Fusion': (results_rank, avg_test_ranks),
}

print(f"{'='*55}")
print(f"  Ensemble Comparison")
print(f"{'='*55}")

best_method = None
best_map3 = 0

for method, (results, _) in ensemble_results.items():
    m3 = results['map3']
    top1 = results['top1_acc']
    print(f"  {method:>20}: MAP@3={m3:.4f}  Top1={top1:.2%}")
    if m3 > best_map3:
        best_map3 = m3
        best_method = method

print(f"\n  Best method: {best_method} with MAP@3 = {best_map3:.4f}")

# Also compare against best individual model
individual_scores = [map_at_3(true_labels, p) for p in all_preds]
best_individual = max(individual_scores)
print(f"  Best individual model MAP@3: {best_individual:.4f}")
print(f"  Ensemble improvement: {best_map3 - best_individual:+.4f}")

# Generate test predictions using the best method
_, best_test_logits = ensemble_results[best_method]
final_test_preds = logits_to_preds(best_test_logits)

# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'prediction': [' '.join(pred) for pred in final_test_preds]
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head(10))

submission.to_csv('submission.csv', index=False)
print(f"\nSaved to submission.csv")

  Ensemble Comparison
        Simple Average: MAP@3=0.5492  Top1=37.65%
          Val-Weighted: MAP@3=0.5497  Top1=37.75%
     Optimized Weights: MAP@3=0.5757  Top1=41.70%
           Rank Fusion: MAP@3=0.5262  Top1=35.30%

  Best method: Optimized Weights with MAP@3 = 0.5757
  Best individual model MAP@3: 0.5749
  Ensemble improvement: +0.0008

Submission shape: (500, 2)
   id prediction
0   1      B E C
1   2      E B D
2   3      E D B
3   4      E A C
4   5      D C E
5   6      B D C
6   7      C D E
7   8      B E A
8   9      C D E
9  10      E A C

Saved to submission.csv


## 16. Log All Results to W&B

In [17]:
PROJECT_NAME = "22f3002548-t22026"

# Log individual models
for i, (name, vs) in enumerate(zip(all_model_names, all_val_scores)):
    train_m3 = map_at_3(true_labels, all_preds[i])
    wandb.init(project=PROJECT_NAME, name=f"m5-model{i+1}", tags=["milestone5", "individual"])
    wandb.log({"model": name, "val_map3": vs, "train_map3": train_m3})
    wandb.finish()

# Log best ensemble
best_results, _ = ensemble_results[best_method]
wandb.init(project=PROJECT_NAME, name="m5-best-ensemble", tags=["milestone5", "ensemble", "final"])
wandb.log({
    "method": best_method,
    "map3": best_results['map3'],
    "top1_accuracy": best_results['top1_acc'],
    "top3_accuracy": best_results['top3_acc'],
    "missed": best_results['missed'],
    "num_models": 5,
    "target_crossed": best_results['map3'] >= 0.75,
})
wandb.finish()

print("All runs logged to W&B!")

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260723_031637-unhbswu5
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run m5-model1
wandb: ⭐️ View project at https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026
wandb: 🚀 View run at https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026/runs/unhbswu5
wandb: updating run metadata; uploading summary
wandb: uploading summary
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: train_map3 ▁
wandb:   val_map3 ▁
wandb: 
wandb: Run summary:
wandb:      model M1: seed42, r16, 256...
wandb: train_map3 0.48225
wandb:   val_map3 0.51167
wandb: 
wandb: 🚀 View run m5-model1 at: https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026/runs/unhbswu5
wandb: ⭐️ View project at: https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wand

All runs logged to W&B!


In [18]:
submission = pd.read_csv('submission.csv')
print(f"Submission: {submission.shape}")
assert submission.shape[0] == len(test_df), "Row count mismatch!"
assert all(len(p.split()) == 3 for p in submission['prediction']), "Need 3 labels per row!"

top1_dist = pd.Series([p.split()[0] for p in submission['prediction']]).value_counts().sort_index()
print(f"\nTop-1 prediction distribution:")
print(top1_dist)
print(f"\n✓ All {len(submission)} rows verified. Ready to submit!")

Submission: (500, 2)

Top-1 prediction distribution:
A    108
B     94
C    114
D     95
E     89
Name: count, dtype: int64

✓ All 500 rows verified. Ready to submit!
